In [ ]:
#@title Setup Environment

!pip install -q safetensors
!pip install -q torch

print("✅ Complete!")

In [ ]:
#@title Conversion Logic
import torch, os, pdb
from safetensors.torch import save_file

def convert_pt_to_safetensors(pt_path, safetensors_path):

  # Load the state dictionary from the .pt file
  # map_location='cpu' is used to load to CPU memory, preventing GPU memory issues
  state_dict = torch.load(
      os.path.join("/content", pt_path),
      map_location=torch.device('cpu'))

  # Save the state dictionary to .safetensors format
  save_file(state_dict['string_to_param'], os.path.join("/content", safetensors_path))
  print(f"Successfully converted '{pt_path}' to '{safetensors_path}'")

def convert_pt_to_safetensors_v2(pt_path, safetensors_path):
    full_pt_path = os.path.join("/content", pt_path)
    # Load with weights_only for security
    state_dict = torch.load(full_pt_path, map_location='cpu', weights_only=False)

    new_dict = {}

    # 1. Handle if the file IS the tensor
    if isinstance(state_dict, torch.Tensor):
        new_dict = {"emb": state_dict}

    # 2. Handle if it's a dictionary (common for most TI embeddings)
    elif isinstance(state_dict, dict):
        # List of keys that might contain the tensor or a sub-dict
        possible_keys = ['string_to_param', 'string_to_token', 'emb', 'embeddings']

        target_data = None
        for key in possible_keys:
            if key in state_dict:
                target_data = state_dict[key]
                break

        if target_data is None:
            # Fallback: if no keys match, just grab the first tensor we find
            for k, v in state_dict.items():
                if isinstance(v, torch.Tensor):
                    target_data = v
                    break

        # If the data we found is ANOTHER dict, we need to find the tensor inside it
        if isinstance(target_data, dict):
            for k, v in target_data.items():
                if isinstance(v, torch.Tensor):
                    new_dict = {"emb": v}
                    break
        elif isinstance(target_data, torch.Tensor):
            new_dict = {"emb": target_data}

    if not new_dict:
        print(f"❌ Failed to find a valid tensor in {pt_path}")
        return

    # Ensure tensor is contiguous (Safetensors requirement)
    new_dict = {k: v.contiguous() for k, v in new_dict.items()}

    save_file(new_dict, os.path.join("/content", safetensors_path))
    print(f"✅ Successfully converted: {safetensors_path}")


print("✅ Complete!")

In [ ]:
#@title Execution

import os
from google.colab import drive

if not os.path.exists('/content/drive'):
  drive.mount('/content/drive')

working_folder = "/content/drive/My Drive/AI/Convert/"
if not os.path.exists(working_folder):
  !mkdir -p "{working_folder}"

with os.scandir(working_folder) as entries:
  for entry in entries:
    if entry.is_file() and entry.name.endswith(".pt"):
        safetensors_full_path = os.path.splitext(entry.path)[0] + ".safetensors"
        print(f"Processing: {entry.path}")
        convert_pt_to_safetensors_v2(entry.path, safetensors_full_path)
        print("---")

print("✅ Complete!")